# Module 1.3 — Generator Metrics: Referenceless

Modules 1.1-1.2 asked "did we fetch the right chunks?" This notebook moves one step further down the pipeline: given whatever was retrieved, **is the generated answer any good** — without needing a hand-written reference answer for every query. These three metrics are the ones you can realistically run against live production traffic, because none of them need a labeled ground truth.

_Source: adapted from `RAG_Evaluation/2.Generator_Evaluation_Metrics.ipynb`, condensed and with hand-constructed test cases in place of the original's live-pipeline dependency (`%run Build_RAG_Pipeline_with_Source.ipynb`) — see Module 1.7 for the live-pipeline version.

## The three metrics, and how they differ

| Metric | Question | Compares against |
|---|---|---|
| **Answer Relevancy** | Does the answer actually address the question asked? | `input` only |
| **Faithfulness** | Is the answer grounded in what was retrieved, with no hallucinated claims? | `retrieval_context` |
| **Hallucination** | Does the answer align with a trusted ground-truth `context`, regardless of what was retrieved? | a separately-supplied `context` (not `retrieval_context`) |

These three are the most commonly confused metrics in this whole space, because "the answer is wrong" can mean any of them failed. The distinguishing question to ask: an answer can be **faithful but irrelevant** (perfectly grounded in the retrieved context, but doesn't actually answer what was asked), **relevant but unfaithful** (directly addresses the question, but invents facts not in the context), or **faithful to bad context** (accurately reflects retrieved chunks that were themselves wrong — Faithfulness can't catch this; that's what Module 1.4's reference-based Answer Correctness is for).

### Answer Relevancy — DeepEval (LLM judge) vs. RAGAS (embedding similarity)

In [ ]:
# ============ ANSWER RELEVANCY ============
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric

query = "What is the time complexity of deleting a node from a binary search tree?"

# A relevant, on-topic answer
relevant_case = LLMTestCase(
    input=query,
    actual_output="Deleting a node from a BST takes O(h) time, where h is the height of the tree.",
)

# An evasive/off-topic answer -- technically about BSTs, but doesn't answer the question asked
evasive_case = LLMTestCase(
    input=query,
    actual_output="Binary search trees are a common data structure used to store sorted data efficiently.",
)

answer_relevancy = AnswerRelevancyMetric(threshold=0.5, model="gpt-4o", include_reason=True)

for label, case in [("Relevant answer", relevant_case), ("Evasive answer (doesn't address the question)", evasive_case)]:
    answer_relevancy.measure(case)
    print(f"{label}: score={answer_relevancy.score:.2f}  reason={answer_relevancy.reason}\n")

DeepEval also has bindings to RAGAS via `deepeval.metrics.ragas.RAGASAnswerRelevancyMetric`, which uses embedding cosine similarity instead of an LLM judge — cheaper and faster, at the cost of missing subtler forms of "technically similar wording, wrong answer":

```python
from deepeval.metrics.ragas import RAGASAnswerRelevancyMetric
from langchain_openai import OpenAIEmbeddings

ragas_relevancy = RAGASAnswerRelevancyMetric(threshold=0.5, model="gpt-4o", embeddings=OpenAIEmbeddings())
ragas_relevancy.measure(relevant_case)
```

### Faithfulness — is the answer grounded in the retrieved context?

In [ ]:
# ============ FAITHFULNESS ============
from deepeval.metrics import FaithfulnessMetric

retrieved_context = [
    "A binary search tree (BST) is a tree data structure where each node has at most two children.",
    "Deleting a node from a BST takes O(h) time, where h is the height of the tree.",
]

# A grounded answer -- everything it claims is supported by retrieved_context
grounded_case = LLMTestCase(
    input="What is the time complexity of deleting a node from a BST?",
    actual_output="Deleting a node from a BST takes O(h) time, where h is the height of the tree.",
    retrieval_context=retrieved_context,
)

# A hallucinated answer -- invents a claim ("self-balancing") that isn't in retrieved_context
hallucinated_case = LLMTestCase(
    input="What is the time complexity of deleting a node from a BST?",
    actual_output="Deleting a node from a BST takes O(log n) time because BSTs automatically self-balance on every deletion.",
    retrieval_context=retrieved_context,
)

faithfulness = FaithfulnessMetric(threshold=0.5, model="gpt-4o", include_reason=True)

for label, case in [("Grounded answer", grounded_case), ("Hallucinated answer (unsupported claim)", hallucinated_case)]:
    faithfulness.measure(case)
    print(f"{label}: score={faithfulness.score:.2f}  reason={faithfulness.reason}\n")

**Reading the output:** the hallucinated case's core number is even correct-*sounding* (still claims a complexity), but it invents an unsupported mechanism ("automatically self-balance on every deletion" — a plain BST does not do this) — exactly the kind of confident, plausible-but-fabricated addition Faithfulness is designed to catch, and exactly the failure mode a plain "is the final number right" check would miss.

### Hallucination Check — does the answer align with a trusted ground-truth context?

In [ ]:
# ============ HALLUCINATION CHECK ============
from deepeval.metrics import HallucinationMetric

# Distinct from `retrieval_context` above: this is a trusted, curated ground-truth document set,
# independent of whatever the retriever happened to fetch for this particular query.
human_ground_truth_context = [
    "Artificial intelligence refers to machines mimicking human intelligence, like problem-solving and learning. "
    "AI includes applications like virtual assistants, robotics, and autonomous vehicles.",
    "Machine learning is a field of artificial intelligence focused on enabling systems to learn patterns from data.",
]

query = "What is AI?"

# An answer consistent with the ground truth
consistent_case = LLMTestCase(
    input=query,
    actual_output="AI refers to machines mimicking human intelligence, used in applications like virtual assistants and robotics.",
    context=human_ground_truth_context,
)

# An answer that contradicts / fabricates against the ground truth
fabricated_case = LLMTestCase(
    input=query,
    actual_output="AI refers to machines mimicking human intelligence to produce cyborgs and electric sheep.",
    context=human_ground_truth_context,
)

hallucination = HallucinationMetric(threshold=0.5, model="gpt-4o", include_reason=True)

for label, case in [("Consistent with ground truth", consistent_case), ("Fabricated claim", fabricated_case)]:
    hallucination.measure(case)
    print(f"{label}: score={hallucination.score:.2f}  reason={hallucination.reason}\n")

**Reading the output:** note the direction HallucinationMetric's score runs — a *higher* score here means *more* hallucination relative to the trusted `context`, the opposite convention from most other DeepEval metrics (where higher = better). Always check a new metric's score direction rather than assuming.

**Faithfulness vs. Hallucination — the distinction that actually matters:** Faithfulness checks the answer against `retrieval_context` (whatever THIS query's retriever happened to fetch). Hallucination checks against a separately-supplied `context` (a trusted, curated ground truth, independent of any one query's retrieval). Use Faithfulness to catch a generator that invents things beyond what it was given; use Hallucination when you have an independent, trusted source of truth to check against regardless of what retrieval returned.

## Summary

- **Answer Relevancy** — does the answer address the question? *(referenceless, `input` only)*
- **Faithfulness** — is the answer grounded in what was retrieved? *(referenceless, checks `retrieval_context`)*
- **Hallucination** — does the answer align with a trusted, independent ground truth? *(needs a curated `context`, higher score = worse — check score direction per metric)*
- None of these three catch "the retrieved context itself was wrong" — an answer can be perfectly faithful to bad information. That gap is exactly what Module 1.4's reference-based Answer Correctness metric is for.
- Next: [Module 1.4](04_Generator_Metrics_Reference_Based.ipynb) covers the reference-based generator metrics — Answer Correctness and Answer Semantic Similarity — plus writing a fully custom `GEval` rubric.